# 02 - Preprocessing\n\nConstrucao da camada silver em Spark SQL, com filtros explicitos, feature engineering e materializacao em Parquet particionado.

In [ ]:
from pyspark.sql import SparkSession\n\nDATA_GLOB = '/data/fhvhv_tripdata_*.parquet'\nLOOKUP_PATH = '/data/taxi_zone_lookup.csv'\nSILVER_OUTPUT = '/data/silver/trips_silver'\n\nspark = (SparkSession.builder\n    .appName('nyc-rideshare-preprocessing')\n    .master('spark://spark-master:7077')\n    .config('spark.executor.memory', '3g')\n    .config('spark.driver.memory', '4g')\n    .config('spark.sql.shuffle.partitions', '200')\n    .getOrCreate())\n\nspark.sparkContext.setLogLevel('WARN')

In [ ]:
spark.read.parquet(DATA_GLOB).createOrReplaceTempView('trips_bronze')\n(spark.read\n    .option('header', True)\n    .option('inferSchema', True)\n    .csv(LOOKUP_PATH)\n    .createOrReplaceTempView('taxi_zone_lookup'))\n\nspark.sql('SELECT COUNT(*) AS total_rows FROM trips_bronze').show()

In [ ]:
spark.sql("""\nCREATE OR REPLACE TEMPORARY VIEW cleaning_bounds AS\nWITH fare_bounds AS (\n    SELECT\n        PERCENTILE_APPROX(base_passenger_fare, 0.001) AS fare_low,\n        PERCENTILE_APPROX(base_passenger_fare, 0.999) AS fare_high\n    FROM trips_bronze\n    WHERE base_passenger_fare > 0\n),\nspeed_bounds AS (\n    SELECT\n        PERCENTILE_APPROX(trip_miles / (trip_time / 3600.0), 0.001) AS speed_low,\n        PERCENTILE_APPROX(trip_miles / (trip_time / 3600.0), 0.999) AS speed_high\n    FROM trips_bronze\n    WHERE trip_miles > 0\n      AND trip_time > 60\n)\nSELECT *\nFROM fare_bounds\nCROSS JOIN speed_bounds\n""")\n\nspark.sql('SELECT * FROM cleaning_bounds').show(truncate=False)

In [ ]:
spark.sql("""
WITH base AS (
    SELECT
        t.*,
        trip_miles / (trip_time / 3600.0) AS speed_mph,
        unix_timestamp(pickup_datetime) - unix_timestamp(request_datetime) AS wait_time_sec,
        pu_zone.Borough AS pu_borough,
        do_zone.Borough AS do_borough
    FROM trips_bronze t
    LEFT JOIN taxi_zone_lookup pu_zone ON t.PULocationID = pu_zone.LocationID
    LEFT JOIN taxi_zone_lookup do_zone ON t.DOLocationID = do_zone.LocationID
)
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN pickup_datetime < request_datetime OR dropoff_datetime <= pickup_datetime THEN 1 ELSE 0 END) AS invalid_temporal_rows,
    SUM(CASE WHEN trip_miles <= 0 OR trip_time < 60 OR trip_time > 14400 THEN 1 ELSE 0 END) AS invalid_physical_rows,
    SUM(CASE WHEN base_passenger_fare <= 0 THEN 1 ELSE 0 END) AS non_positive_fare_rows,
    SUM(CASE WHEN PULocationID IN (264, 265) OR DOLocationID IN (264, 265) THEN 1 ELSE 0 END) AS invalid_zone_rows,
    SUM(CASE WHEN wait_time_sec < 0 THEN 1 ELSE 0 END) AS negative_wait_rows,
    SUM(CASE WHEN pu_borough IS NULL OR do_borough IS NULL THEN 1 ELSE 0 END) AS null_borough_rows,
    SUM(CASE WHEN airport_fee IS NULL THEN 1 ELSE 0 END) AS null_airport_fee_rows
FROM base
""").show(truncate=False)

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW trips_silver AS
WITH base AS (
    SELECT
        t.*,
        trip_miles / (trip_time / 3600.0) AS speed_mph,
        unix_timestamp(pickup_datetime) - unix_timestamp(request_datetime) AS wait_time_sec
    FROM trips_bronze t
),
joined_zones AS (
    SELECT
        b.*,
        pu_zone.Borough AS pu_borough,
        do_zone.Borough AS do_borough
    FROM base b
    LEFT JOIN taxi_zone_lookup pu_zone
      ON b.PULocationID = pu_zone.LocationID
    LEFT JOIN taxi_zone_lookup do_zone
      ON b.DOLocationID = do_zone.LocationID
)
SELECT
    hvfhs_license_num,
    dispatching_base_num,
    originating_base_num,
    request_datetime,
    pickup_datetime,
    dropoff_datetime,
    PULocationID,
    DOLocationID,
    trip_miles,
    trip_time,
    base_passenger_fare,
    tolls,
    bcf,
    sales_tax,
    congestion_surcharge,
    airport_fee,
    tips,
    driver_pay,
    pu_borough,
    do_borough,
    HOUR(pickup_datetime) AS pickup_hour,
    DAYOFWEEK(pickup_datetime) AS pickup_dow,
    MONTH(pickup_datetime) AS pickup_month_num,
    YEAR(pickup_datetime) AS pickup_year,
    DATE_FORMAT(pickup_datetime, 'yyyy-MM') AS pickup_year_month,
    CASE WHEN DAYOFWEEK(pickup_datetime) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend,
    CASE WHEN HOUR(pickup_datetime) BETWEEN 7 AND 9 OR HOUR(pickup_datetime) BETWEEN 16 AND 19 THEN 1 ELSE 0 END AS is_rush_hour,
    CASE WHEN HOUR(pickup_datetime) >= 22 OR HOUR(pickup_datetime) <= 4 THEN 1 ELSE 0 END AS is_late_night,
    CASE WHEN PULocationID IN (1, 132, 138) THEN 1 ELSE 0 END AS pickup_airport,
    CASE WHEN DOLocationID IN (1, 132, 138) THEN 1 ELSE 0 END AS dropoff_airport,
    -- NULL-safe: zona desconhecida cai em 0 (pessimista, evita finger-pointing falso)
    CASE
        WHEN pu_borough IS NULL OR do_borough IS NULL THEN 0
        WHEN pu_borough = do_borough THEN 1
        ELSE 0
    END AS same_borough,
    CASE WHEN shared_request_flag = 'Y' THEN 1 ELSE 0 END AS shared_req,
    CASE WHEN wav_request_flag = 'Y' THEN 1 ELSE 0 END AS wav_req,
    wait_time_sec,
    speed_mph
FROM joined_zones
CROSS JOIN cleaning_bounds cb
WHERE pickup_datetime >= request_datetime              -- consistencia temporal
  AND dropoff_datetime > pickup_datetime               -- corrida concluida
  AND trip_miles > 0                                   -- distancia positiva
  AND trip_time BETWEEN 60 AND 14400                   -- duracao entre 1 min e 4 h
  AND wait_time_sec >= 0                               -- espera nao negativa
  AND base_passenger_fare BETWEEN cb.fare_low AND cb.fare_high
  AND speed_mph BETWEEN cb.speed_low AND cb.speed_high -- percentis para cauda pesada
  AND PULocationID NOT IN (264, 265)                   -- Unknown / Outside NYC
  AND DOLocationID NOT IN (264, 265)
""")

spark.sql('SELECT COUNT(*) AS silver_rows FROM trips_silver').show()

In [ ]:
spark.sql("""\nWITH total AS (SELECT COUNT(*) AS total_rows FROM trips_bronze),\nsilver AS (SELECT COUNT(*) AS silver_rows FROM trips_silver)\nSELECT\n    total.total_rows,\n    silver.silver_rows,\n    ROUND(100.0 * (total.total_rows - silver.silver_rows) / total.total_rows, 4) AS pct_rows_removed\nFROM total\nCROSS JOIN silver\n""").show(truncate=False)

In [ ]:
(spark.table('trips_silver')\n    .write\n    .mode('overwrite')\n    .partitionBy('pickup_year_month')\n    .parquet(SILVER_OUTPUT))\n\nprint(f'Silver materializada em {SILVER_OUTPUT}')

In [ ]:
spark.read.parquet(SILVER_OUTPUT).createOrReplaceTempView('trips_silver_disk')\nspark.sql("""\nSELECT pickup_year_month, COUNT(*) AS trips\nFROM trips_silver_disk\nGROUP BY 1\nORDER BY 1\n""").show(100, truncate=False)

In [ ]:
spark.stop()